# Airfoil: geometry, surrogate prediction, validation against the stored CFD

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PINNeAPPle-Labs/PINNeAPPle/blob/main/notebooks/colab/02_airfoil_surrogate.ipynb)

A 2-D airfoil at Reynolds number 4 million. The shape is six CST weights (Kulfan 2008); a trained ensemble of
networks maps (shape, angle of attack) to the lift, drag and pitching-moment coefficients with a spread. The
surrogate was trained on a 436-run OpenFOAM campaign (k-omega SST); the four NACA reference airfoils were held out.

1. **Geometry**: the NACA 2412 as CST weights, its thickness and camber, and three variants.
2. **Prediction**: Cl, Cd, Cm and L/D at 4 degrees, with the ensemble spread.
3. **Validation**: against OpenFOAM runs stored in the repository (the baseline and the three variants, not in the
   training set), against XFOIL, and against two physics checks (thin-airfoil zero-lift angle and lift-curve slope),
   summarised in a trust card.

The stored model and results come from the repository (`apps/aero_optimizer/model/`,
`examples/handoff_cases/airfoil_ld/`): read locally from a checkout, otherwise downloaded from GitHub.

In [ ]:
# Setup: on Colab (or wherever PINNeAPPle is missing) install it from PyPI, then pick the device.
import importlib.util, os, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB or importlib.util.find_spec("pinneapple") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pinneapple>=0.7.0"], check=True)

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import pinneapple as pp

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PINNeAPPle {pp.__version__} on {DEVICE}")

In [ ]:
# Parameters. SMOKE=True keeps the sizes minimal (the CI runs it that way); this notebook is fast either way.
SMOKE = os.environ.get("PINNEAPPLE_NB_SMOKE", "0") == "1"
ALPHA = 4.0                                    # angle of attack [deg] of the design point
POLAR = np.arange(-2.0, 9.0, 2.0 if SMOKE else 0.5)  # angles of the predicted polar [deg]

In [ ]:
from pathlib import Path
import urllib.request

REPO_RAW = "https://raw.githubusercontent.com/PINNeAPPle-Labs/PINNeAPPle/main/"

def repo_file(rel):
    """A file of the PINNeAPPle repository: from the local checkout if there is one, else downloaded from GitHub."""
    roots = [Path(os.environ["PINNEAPPLE_REPO"])] if os.environ.get("PINNEAPPLE_REPO") else []
    for root in roots + [Path.cwd(), *Path.cwd().parents]:
        if (root / rel).is_file() and (root / "pinneapple").is_dir():
            return root / rel
    dest = Path("pinneapple_files") / rel
    if not dest.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

CASE = "examples/handoff_cases/airfoil_ld/"

## 1. Geometry

`pinneapple_design.aero.geometry` fits CST weights to any NACA 4-digit section and reports its properties; `valid`
rejects crossing or wavy surfaces before anything is predicted for them.

In [ ]:
from pinneapple_design.aero.geometry import BOUNDS, PARAMS, naca4, properties, surfaces, valid

cst = json.loads(repo_file(CASE + "geometry/cst_parameters.json").read_text())
designs = {"NACA 2412 (baseline)": naca4("2412")}
variants = pd.read_csv(repo_file(CASE + "results/variants/variants_alpha4.csv"))
for _, r in variants.iloc[1:].iterrows():                  # the three variants, CST weights as run in OpenFOAM
    designs[r["variant"]] = r[list(PARAMS)].to_numpy(float)

xc = np.linspace(0, 1, 201)
fig, ax = plt.subplots(figsize=(9, 2.8))
for name, w in designs.items():
    ok, why = valid(w)
    p = properties(w)
    yu, yl = surfaces(w, xc)
    ax.plot(np.r_[xc, xc[::-1]], np.r_[yu, yl[::-1]], label=f"{name}: t/c {p['thickness']:.3f}, "
            f"camber {p['camber']:.3f}" + ("" if ok else f" (invalid: {why})"))
ax.set(aspect="equal", xlabel="x/c", ylabel="y/c")
ax.legend(fontsize=8, loc="upper right")
plt.show()
print("CST weights of the NACA 2412 fit:", dict(zip(PARAMS, np.round(designs["NACA 2412 (baseline)"], 4))))
print("as run in OpenFOAM (case package):", dict(zip(PARAMS, np.round(variants.iloc[0][list(PARAMS)].to_numpy(float), 4))))
outside = {n: [p for p, v, (lo, hi) in zip(PARAMS, w, BOUNDS) if not lo <= v <= hi] for n, w in designs.items()}
print("weights outside the surrogate's training bounds:", {n: p for n, p in outside.items() if p} or "none")

## 2. Surrogate prediction

The trained bundle holds an ensemble of five networks (the spread is the uncertainty) and the normalisation of its
targets. `Engine.mlp_coefficients(shapes, alphas)` returns the mean and the standard deviation of each coefficient.

> **Model zoo.** Once `pp.zoo` is released this cell becomes `model = pp.zoo.load("airfoil-cst-mlp-ensemble")`
> (tracked in the follow-up issue of the zoo); until then the bundle is read from the repository.

In [ ]:
from pinneapple_design.aero.optimize import Engine

USE_ZOO = hasattr(pp, "zoo")       # switch to the model zoo when it exists
if USE_ZOO:
    engine = pp.zoo.load("airfoil-cst-mlp-ensemble")
else:
    engine = Engine.load(str(repo_file("apps/aero_optimizer/model/surrogate.pt")))
print("ensemble members:", len(engine.mlp.nets))

In [ ]:
S = np.array(list(designs.values()))
pred = engine.mlp_coefficients(S, np.array([ALPHA]))
table = pd.DataFrame({"design": list(designs),
                      "Cl": pred["cl"][:, 0], "Cl std": pred["cl_std"][:, 0],
                      "Cd": pred["cd"][:, 0], "Cd std": pred["cd_std"][:, 0],
                      "Cm": pred["cm"][:, 0], "L/D": pred["cl"][:, 0] / pred["cd"][:, 0]})
table.round(5)

## 3. Validation against the stored reference

Each design was run in OpenFOAM (simpleFoam, k-omega SST, the campaign's medium grid) and stored in
`results/variants/variants_alpha4.csv`. For the baseline there is also XFOIL with transition tripped at 1 % chord
(the closest model to a fully turbulent RANS run).

In [ ]:
cfd = variants.set_index("variant")
ref = pd.DataFrame({"design": list(designs),
                    "Cl CFD": cfd["Cl"].to_numpy(), "Cd CFD": cfd["Cd"].to_numpy(), "Cm CFD": cfd["Cm"].to_numpy()})
cmp = table.merge(ref, on="design")
cmp["Cl err %"] = 100 * (cmp["Cl"] - cmp["Cl CFD"]) / cmp["Cl CFD"]
cmp["Cd err %"] = 100 * (cmp["Cd"] - cmp["Cd CFD"]) / cmp["Cd CFD"]
cmp["Cl err / std"] = np.abs(cmp["Cl"] - cmp["Cl CFD"]) / cmp["Cl std"]
cmp[["design", "Cl", "Cl CFD", "Cl err %", "Cd", "Cd CFD", "Cd err %", "Cl err / std"]].round(4)

In [ ]:
xfoil = json.loads(repo_file(CASE + "results/reference/xfoil_summary.json").read_text())
print(json.dumps(xfoil, indent=1)[:600])

### Physics checks of the predicted polar

Two checks that do not use any CFD number:

* **zero-lift angle**: thin-airfoil theory gives $-2.08^\circ$ for the NACA 2412 mean line;
* **lift-curve slope**: thin-airfoil theory gives $2\pi$ per radian (0.110 per degree); viscous sections at this
  Reynolds number sit a few per cent below it.

In [ ]:
polar = engine.mlp_coefficients(designs["NACA 2412 (baseline)"][None, :], POLAR)
cl = polar["cl"][0]
lin = (POLAR >= -2) & (POLAR <= 6)                          # linear range
slope, cl0 = np.polyfit(POLAR[lin], cl[lin], 1)
alpha_zl = -cl0 / slope
print(f"lift slope {slope:.4f} /deg (thin airfoil 0.1097), zero-lift angle {alpha_zl:.2f} deg (theory -2.08)")

camp = pd.read_csv(repo_file(CASE + "results/campaign/campaign_runs.csv"))
naca = camp[(camp["shape_id"] == "NACA 2412") & (camp["status"] == "converged")].sort_values("alpha_deg")
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
axs[0].fill_between(POLAR, cl - 2 * polar["cl_std"][0], cl + 2 * polar["cl_std"][0], alpha=0.3, label="surrogate ±2 std")
axs[0].plot(POLAR, cl, label="surrogate")
axs[0].plot(naca["alpha_deg"], naca["Cl"], "ko", label="OpenFOAM (held out)")
axs[0].set(xlabel="alpha [deg]", ylabel="Cl")
axs[1].plot(POLAR, polar["cd"][0], label="surrogate")
axs[1].plot(naca["alpha_deg"], naca["Cd"], "ko", label="OpenFOAM (held out)")
axs[1].set(xlabel="alpha [deg]", ylabel="Cd")
for a in axs:
    a.legend()
plt.tight_layout()
plt.show()

### The trust card

`TrustReport` (`pinneapple_analysis.trust`, in every release) turns the checks into a decision: a failing critical
check rejects, any other failure asks for review.

In [ ]:
from pinneapple_analysis.trust import Check, TrustReport

base = cmp.iloc[0]
inside = bool(np.all((S >= BOUNDS[:, 0]) & (S <= BOUNDS[:, 1])))
checks = [
    Check("geometry valid (no crossing or wavy surface)", "supports" if all(valid(w)[0] for w in S) else "contradicts",
          critical=True),
    Check("designs inside the training bounds of the CST weights (OOD)", "supports" if inside else "contradicts"),
    Check("baseline Cl vs OpenFOAM, relative error", "supports" if abs(base["Cl err %"]) <= 3 else "contradicts",
          critical=True, value=abs(base["Cl err %"]) / 100, limit=0.03, source="variants_alpha4.csv"),
    Check("baseline Cd vs OpenFOAM, relative error", "supports" if abs(base["Cd err %"]) <= 5 else "contradicts",
          critical=True, value=abs(base["Cd err %"]) / 100, limit=0.05, source="variants_alpha4.csv"),
    Check("variants Cl vs OpenFOAM, max relative error", "supports" if cmp["Cl err %"].abs().max() <= 5 else "contradicts",
          value=cmp["Cl err %"].abs().max() / 100, limit=0.05),
    Check("zero-lift angle vs thin-airfoil theory (|diff| <= 0.5 deg)",
          "supports" if abs(alpha_zl + 2.08) <= 0.5 else "contradicts", value=float(alpha_zl), limit=-2.08),
    Check("lift-curve slope within 15 % of 2 pi", "supports" if abs(slope / 0.1097 - 1) <= 0.15 else "contradicts",
          value=float(slope), limit=0.1097),
    Check("ensemble spread covers the Cl error (|err| <= 3 std)",
          "supports" if (cmp["Cl err / std"] <= 3).all() else "contradicts", value=float(cmp["Cl err / std"].max()), limit=3.0),
]
report = TrustReport(checks, model="airfoil MLP ensemble")
print("decision:", report.decisao)
for c in checks:
    print(f"  [{'PASS' if c.status == 'supports' else 'FAIL'}] {c.name}" + (f"  value {c.value:.4g}" if c.value is not None else ""))

The applicability check is the one that fails here: the NACA 4412 variant has a mid-chord camber weight (0.142)
just above the largest value in the training campaign (0.14). Its prediction happens to be close to the CFD, but
the surrogate gives no guarantee there, so the card asks for review instead of approving. Read the card, not just
the error table: if the spread check fails, the surrogate is more confident than it should
be, and an optimiser using it would trust designs it should not. The case package
(`examples/handoff_cases/airfoil_ld/`) runs the optimum back in OpenFOAM for exactly this reason.